# 2.3.1. Загрузка данных и первичный анализ

## 1.Импорт библиотек

In [40]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix

from sklearn.dummy import DummyClassifier

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV

from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text

from sklearn.ensemble import RandomForestClassifier

from sklearn.ensemble import AdaBoostClassifier

from sklearn.inspection import permutation_importance

## 2.Загрузка CSV

In [41]:
df = pd.read_csv('data/S06-hw-dataset-02.csv')

## 3.Анализ

### 3.1.Базовые статистики

In [42]:
print(df.head(), df.info(), df.describe(), sep = '\n')

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18000 entries, 0 to 17999
Data columns (total 39 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   id       18000 non-null  int64  
 1   f01      18000 non-null  float64
 2   f02      18000 non-null  float64
 3   f03      18000 non-null  float64
 4   f04      18000 non-null  float64
 5   f05      18000 non-null  float64
 6   f06      18000 non-null  float64
 7   f07      18000 non-null  float64
 8   f08      18000 non-null  float64
 9   f09      18000 non-null  float64
 10  f10      18000 non-null  float64
 11  f11      18000 non-null  float64
 12  f12      18000 non-null  float64
 13  f13      18000 non-null  float64
 14  f14      18000 non-null  float64
 15  f15      18000 non-null  float64
 16  f16      18000 non-null  float64
 17  f17      18000 non-null  float64
 18  f18      18000 non-null  float64
 19  f19      18000 non-null  float64
 20  f20      18000 non-null  float64
 21  f21      180

### 3.2.Распределение

In [43]:
df.target.value_counts(normalize=True)

target
0    0.737389
1    0.262611
Name: proportion, dtype: float64

## 4.Разделение

In [44]:
X, y = df.iloc[:,1:-1], df.target

# 2.3.2. Train/Test-сплит и воспроизводимость

## 1.Разделение на train/test

In [45]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42, stratify = y)

## 2.Пояснение
random_state необходим для одинакового разделения при каждом запуске, значение особой роли не играет. stratify необходим для разделения классов в train/test в той пропорции, в какой был исходный датасет

# 2.3.3. Baseline’ы

In [46]:
dm = DummyClassifier(strategy="most_frequent")
dm.fit(X_train, y_train)

y_pred = dm.predict(X_test)
y_proba = dm.predict_proba(X_test)[:,1]

acc = accuracy_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_proba)
f1 = f1_score(y_test, y_pred)
c_matrix = confusion_matrix(y_test, y_pred)

print(f'accuracy_score: {acc}\nroc_auc_score: {roc_auc}\nf1_score: {f1}\nconfusion_matrix:\n {c_matrix}')

accuracy_score: 0.7375
roc_auc_score: 0.5
f1_score: 0.0
confusion_matrix:
 [[2655    0]
 [ 945    0]]


In [47]:
pipe = Pipeline([
    ('scaller', StandardScaler()),
    ('logreg', LogisticRegression(max_iter = 1000, random_state = 42))
    ])

param_grid = {'logreg__C': [0.01, 0.1, 1.0, 10.0 ,100.0]}

grid_search = GridSearchCV(
    pipe,
    param_grid,
    cv = 5,
    scoring = 'roc_auc',
    n_jobs = -1)

grid_search.fit(X_train, y_train)

lrm = grid_search.best_estimator_

y_pred = lrm.predict(X_test)
y_proba = lrm.predict_proba(X_test)[:,1]

acc = accuracy_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_proba)
f1 = f1_score(y_test, y_pred)
c_matrix = confusion_matrix(y_test, y_pred)

print(f'accuracy_score: {acc}\nroc_auc_score: {roc_auc}\nf1_score: {f1}\nconfusion_matrix:\n {c_matrix}')

accuracy_score: 0.8119444444444445
roc_auc_score: 0.7976938789744817
f1_score: 0.5606748864373783
confusion_matrix:
 [[2491  164]
 [ 513  432]]


# 2.3.4. Модели недели 6

## 1.Дерево

In [48]:
tfm = DecisionTreeClassifier(random_state = 42)
tfm.fit(X_train, y_train)

path = tfm.cost_complexity_pruning_path(X_train, y_train)
ccp_alphas = path.ccp_alphas

ccp_alphas = np.unique(ccp_alphas)
ccp_alphas = ccp_alphas[:: max(1,int(0.1*(len(ccp_alphas))))]

tm_data = {'model': None,
            'accuracy': None,
            'roc_auc': -1.0,
            'f1': None,
            'confusion_matrix': None,
            'node_count': None
            }
            
for a in ccp_alphas:
    # Чем больше ccp_alpha → тем сильнее обрезка (дерево меньше).
    m = DecisionTreeClassifier(random_state=42, ccp_alpha=float(a))
    m.fit(X_train, y_train)
    
    cur_roc_auc = roc_auc_score(y_test, m.predict_proba(X_test)[:,1])
    if cur_roc_auc>tm_data['roc_auc']:
        y_pred =  m.predict(X_test)
        tm_data['model'] = m
        tm_data['accuracy'] = accuracy_score(y_test, y_pred)
        tm_data['roc_auc'] = cur_roc_auc
        tm_data['f1'] = f1_score(y_test, y_pred)
        tm_data['confusion_matrix'] = confusion_matrix(y_test, y_pred)
        tm_data['node_count'] = m.tree_.node_count

print(f'accuracy_score: {tm_data['accuracy']}\nroc_auc_score: { tm_data['roc_auc']}\nf1_score: { tm_data['f1']}\nconfusion_matrix:\n { tm_data['confusion_matrix']}\nnode_count: {tm_data['node_count']}')

accuracy_score: 0.8405555555555555
roc_auc_score: 0.8399553602566784
f1_score: 0.6631455399061033
confusion_matrix:
 [[2461  194]
 [ 380  565]]
node_count: 279


## 2.Лес

In [49]:
rfm_data = {'model': None,
            'accuracy': None,
            'roc_auc': -1.0,
            'f1': None,
            'confusion_matrix': None,
            'oob_score': None,
            }

for mf in [0.3, 0.5, 0.7, 1.0, 'sqrt', 'log2']:
    rfm = RandomForestClassifier(
        n_estimators = 100,
        random_state = 42,
        max_features = mf,
        oob_score = True,
        n_jobs = -1
        ) 
    rfm.fit(X_train, y_train)
    
    y_proba = rfm.predict_proba(X_test)[:,1]
    cur_roc_auc = roc_auc_score(y_test,y_proba)
    if cur_roc_auc>rfm_data['roc_auc']:
        y_pred =  rfm.predict(X_test)
        rfm_data['model'] = rfm
        rfm_data['accuracy'] = accuracy_score(y_test, y_pred)
        rfm_data['roc_auc'] = cur_roc_auc
        rfm_data['f1'] = f1_score(y_test, y_pred)
        rfm_data['confusion_matrix'] = confusion_matrix(y_test, y_pred)
        rfm_data['oob_score'] = rfm.oob_score_

for key, val in rfm_data.items():
    if key != 'model':
        print(key, val)

accuracy 0.8905555555555555
roc_auc 0.9261519146264909
f1 0.7555831265508685
confusion_matrix [[2597   58]
 [ 336  609]]
oob_score 0.8901388888888889


## 3.boosting 

In [50]:
tree = DecisionTreeClassifier(random_state = 42, max_depth = 1)

ada = AdaBoostClassifier(estimator = tree,
                         n_estimators = 200,
                         learning_rate = 0.6,
                         random_state = 42
                        )

ada.fit(X_train, y_train)

y_proba = ada.predict_proba(X_test)[:,1]
y_pred =  ada.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test,y_proba)
f1 = f1_score(y_test, y_pred)
c_matrix = confusion_matrix(y_test, y_pred)

print(f'accuracy_score: {accuracy}\nroc_auc_score: {roc_auc}\nf1_score: {f1}\nconfusion_matrix:\n {c_matrix}')

accuracy_score: 0.8108333333333333
roc_auc_score: 0.8208549307984336
f1_score: 0.5260960334029228
confusion_matrix:
 [[2541  114]
 [ 567  378]]


# 2.3.6. Интерпретация

In [57]:
perm = permutation_importance(
    rfm_data['model'], X_test, y_test,
    n_repeats=8,
    random_state=42,
    scoring="roc_auc"
)

pimp = perm.importances_mean
idx = np.argsort(pimp)[::-1][:12]
for i in idx:
    print(X.columns[i])

f16
f01
f19
f12
f07
f23
f02
f30
f08
f18
f13
f05


# 2.4. Артефакты эксперимента